In [ ]:

%pip install -q pennylane torch scikit-learn pandas numpy matplotlib


In [ ]:

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pennylane as qml
import random
import time
import zipfile

from pathlib import Path

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, recall_score,
    precision_score, f1_score,
    confusion_matrix
)

SEED = 42
N_QUBITS = 4
N_LAYERS = 2
EPOCHS = 30
LR = 0.03
RESIDUAL_SCALE = 0.5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_default_dtype(torch.float64)

OUTPUT_DIR = Path("phase_8_results")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Phase 8 ready.")


Phase 8 ready.


In [ ]:

from google.colab import files

if not Path(
    "limited_data/data_n40_seed42.npz"
).exists():

    uploaded = files.upload()

    zip_name = next(
        name for name in uploaded
        if name.endswith(".zip")
    )

    with zipfile.ZipFile(zip_name) as archive:
        archive.extractall(".")

dataset = load_breast_cancer()

X_raw = dataset.data
y_all = (dataset.target == 0).astype(int)

with np.load(
    "limited_data/data_n40_seed42.npz"
) as data:
    subset_idx = data["train_idx"].copy()

X_subset = X_raw[subset_idx]
y_subset = y_all[subset_idx]

print("Training subset:", X_subset.shape)
print("Malignant:", sum(y_subset == 1))
print("Benign:", sum(y_subset == 0))


Saving qbreast_limited_data.zip to qbreast_limited_data.zip
Training subset: (40, 30)
Malignant: 15
Benign: 25


In [ ]:
#Phase 8.4 — Training and Validation Split


X_train_raw, X_val_raw, y_train, y_val = (
    train_test_split(
        X_subset,
        y_subset,
        test_size=0.25,
        stratify=y_subset,
        random_state=SEED
    )
)

def make_preprocessor():
    return Pipeline([
        ("standardize", StandardScaler()),
        ("pca", PCA(n_components=4)),
        ("angles", MinMaxScaler(
            feature_range=(0, np.pi),
            clip=True
        ))
    ])

preprocessor = make_preprocessor()

X_train = preprocessor.fit_transform(
    X_train_raw
)

X_val = preprocessor.transform(
    X_val_raw
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)


Training: (30, 4)
Validation: (10, 4)


In [ ]:
#Phase 8.5 — Residual Network Components


class ClassicalBase(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(4, 4),
            nn.Tanh(),
            nn.Linear(4, 1)
        )

    def forward(self, x):
        return self.network(x).squeeze(-1)


class ClassicalResidualBranch(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(4, 3),
            nn.Tanh(),
            nn.Linear(3, 1)
        )

    def forward(self, x):
        return self.network(x).squeeze(-1)


class QuantumResidualBranch(nn.Module):
    def __init__(self):
        super().__init__()

        dev = qml.device(
            "default.qubit",
            wires=N_QUBITS
        )

        @qml.qnode(
            dev,
            interface="torch",
            diff_method="backprop"
        )
        def circuit(inputs, weights):

            for i in range(N_QUBITS):
                qml.RY(inputs[i], wires=i)

            for layer in range(N_LAYERS):

                for i in range(N_QUBITS):
                    qml.RY(
                        weights[layer, i, 0],
                        wires=i
                    )
                    qml.RZ(
                        weights[layer, i, 1],
                        wires=i
                    )

                for i in range(N_QUBITS - 1):
                    qml.CNOT(
                        wires=[i, i + 1]
                    )

            return [
                qml.expval(qml.PauliZ(i))
                for i in range(N_QUBITS)
            ]

        self.quantum = qml.qnn.TorchLayer(
            circuit,
            {
                "weights": (
                    N_LAYERS, N_QUBITS, 2
                )
            }
        )

        self.output = nn.Linear(
            N_QUBITS, 1
        )

    def forward(self, x):
        outputs = torch.stack([
            self.quantum(sample)
            for sample in x
        ])
        return self.output(outputs).squeeze(-1)


In [ ]:
#Phase 8.6 — Residual Classifiers

class ResidualClassifier(nn.Module):
    def __init__(self, branch_type="none"):
        super().__init__()

        self.base = ClassicalBase()
        self.branch_type = branch_type

        if branch_type == "classical":
            self.residual = ClassicalResidualBranch()

        elif branch_type == "quantum":
            self.residual = QuantumResidualBranch()

        elif branch_type == "none":
            self.residual = None

        else:
            raise ValueError(branch_type)

    def forward(self, x):
        base_logit = self.base(x)

        if self.residual is None:
            return base_logit

        correction = self.residual(x)

        return (
            base_logit
            + RESIDUAL_SCALE * correction
        )


MODEL_TYPES = {
    "Classical MLP": "none",
    "Classical Residual": "classical",
    "Quantum Residual": "quantum"
}


In [ ]:
#Phase 8.7 — Residual Architecture Validation

sample_x = torch.tensor(
    X_train[:4],
    dtype=torch.float64
)

sample_y = torch.tensor(
    y_train[:4],
    dtype=torch.float64
)

for name, branch_type in MODEL_TYPES.items():

    torch.manual_seed(SEED)

    model = ResidualClassifier(
        branch_type
    ).double()

    params = sum(
        p.numel()
        for p in model.parameters()
    )

    logits = model(sample_x)

    assert logits.shape == (4,)

    loss = nn.BCEWithLogitsLoss()(
        logits, sample_y
    )

    loss.backward()

    print(name, "parameters:", params)

    if branch_type == "quantum":
        grad_norm = (
            model.residual.quantum.weights
            .grad.norm().item()
        )

        print(
            "Quantum gradient norm:",
            grad_norm
        )

        assert np.isfinite(grad_norm)
        assert grad_norm > 1e-10

print("All architecture checks passed.")


Classical MLP parameters: 25
Classical Residual parameters: 44
Quantum Residual parameters: 46
Quantum gradient norm: 0.06042698554024021
All architecture checks passed.


In [ ]:
#Phase 8.8 — Train Residual Models

def train_residual_model(
    model_name,
    Xtr, ytr,
    Xval, yval,
    seed=42,
    epochs=30,
    lr=0.03
):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    model = ResidualClassifier(
        MODEL_TYPES[model_name]
    ).double()

    Xtr_t = torch.tensor(
        Xtr, dtype=torch.float64
    )

    Xval_t = torch.tensor(
        Xval, dtype=torch.float64
    )

    ytr_t = torch.tensor(
        ytr, dtype=torch.float64
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )

    criterion = nn.BCEWithLogitsLoss()

    start = time.perf_counter()

    for epoch in range(epochs):

        model.train()
        optimizer.zero_grad()

        logits = model(Xtr_t)
        loss = criterion(logits, ytr_t)

        loss.backward()
        optimizer.step()

    elapsed = time.perf_counter() - start

    model.eval()

    with torch.no_grad():
        val_logits = model(Xval_t)

        probs = torch.sigmoid(
            val_logits
        ).cpu().numpy()

        final_train_loss = criterion(
            model(Xtr_t),
            ytr_t
        ).item()

    pred = (probs >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        yval, pred,
        labels=[0, 1]
    ).ravel()

    return {
        "roc_auc": roc_auc_score(
            yval, probs
        ),
        "recall": recall_score(
            yval, pred,
            zero_division=0
        ),
        "precision": precision_score(
            yval, pred,
            zero_division=0
        ),
        "f1": f1_score(
            yval, pred,
            zero_division=0
        ),
        "specificity": tn / (tn + fp),
        "false_negatives": int(fn),
        "false_positives": int(fp),
        "final_train_loss": final_train_loss,
        "training_seconds": elapsed
    }


In [ ]:
#Phase 8.9 — Classical vs Quantum Residual Pilot

pilot_records = []

for model_name in MODEL_TYPES:

    result = train_residual_model(
        model_name,
        X_train,
        y_train,
        X_val,
        y_val,
        seed=SEED,
        epochs=EPOCHS,
        lr=LR
    )

    pilot_records.append({
        "model": model_name,
        **result
    })

    print(
        model_name,
        "AUC:",
        round(result["roc_auc"], 4),
        "Recall:",
        round(result["recall"], 4)
    )

pilot_df = pd.DataFrame(pilot_records)

display(pilot_df.round(4))

pilot_df.to_csv(
    OUTPUT_DIR / "phase8_pilot.csv",
    index=False
)


Classical MLP AUC: 1.0 Recall: 0.5
Classical Residual AUC: 1.0 Recall: 0.5
Quantum Residual AUC: 1.0 Recall: 0.5


,model,roc_auc,recall,precision,f1,specificity,false_negatives,false_positives,final_train_loss,training_seconds
0,Classical MLP,1.0,0.5,1.0,0.6667,1.0,2,0,0.2958,0.0279
1,Classical Residual,1.0,0.5,1.0,0.6667,1.0,2,0,0.1952,0.0288
2,Quantum Residual,1.0,0.5,1.0,0.6667,1.0,2,0,0.2577,10.0667


In [ ]:
#8.10.1 — Repeated Residual Network Benchmark

P810_SIZES = [40, 80, 160]
P810_SEEDS = [42, 123, 2026]

P810_EPOCHS = 30
P810_LR = 0.03

P810_MODELS = [
    "Classical MLP",
    "Classical Residual",
    "Quantum Residual"
]

P810_DIR = Path("phase_8_results")
P810_DIR.mkdir(exist_ok=True)

print("Phase 8.10 ready.")


Phase 8.10 ready.


In [ ]:

from sklearn.model_selection import StratifiedKFold

p810_records = []

for n_samples in P810_SIZES:
    for seed in P810_SEEDS:

        path = Path(
            f"limited_data/data_n{n_samples}_seed{seed}.npz"
        )

        with np.load(path) as data:
            subset_idx = data["train_idx"].copy()

        X_subset = X_raw[subset_idx]
        y_subset = y_all[subset_idx]

        cv = StratifiedKFold(
            n_splits=3,
            shuffle=True,
            random_state=seed
        )

        for fold, (tr, val) in enumerate(
            cv.split(X_subset, y_subset)
        ):

            prep = make_preprocessor()

            Xtr = prep.fit_transform(
                X_subset[tr]
            )

            Xval = prep.transform(
                X_subset[val]
            )

            ytr = y_subset[tr]
            yval = y_subset[val]

            for model_name in P810_MODELS:

                result = train_residual_model(
                    model_name,
                    Xtr,
                    ytr,
                    Xval,
                    yval,
                    seed=seed + fold,
                    epochs=P810_EPOCHS,
                    lr=P810_LR
                )

                p810_records.append({
                    "n_samples": n_samples,
                    "seed": seed,
                    "fold": fold,
                    "model": model_name,
                    **result
                })

                print(
                    f"n={n_samples}, "
                    f"seed={seed}, "
                    f"fold={fold}, "
                    f"{model_name}: "
                    f"AUC={result['roc_auc']:.4f}, "
                    f"Recall={result['recall']:.4f}"
                )

        # Save completed seed
        pd.DataFrame(p810_records).to_csv(
            P810_DIR / "phase810_raw.csv",
            index=False
        )

p810_df = pd.DataFrame(p810_records)

print("Phase 8.10 completed.")


n=40, seed=42, fold=0, Classical MLP: AUC=1.0000, Recall=0.8000
n=40, seed=42, fold=0, Classical Residual: AUC=1.0000, Recall=0.8000
n=40, seed=42, fold=0, Quantum Residual: AUC=1.0000, Recall=0.8000
n=40, seed=42, fold=1, Classical MLP: AUC=0.8250, Recall=0.6000
n=40, seed=42, fold=1, Classical Residual: AUC=0.8250, Recall=0.6000
n=40, seed=42, fold=1, Quantum Residual: AUC=0.8500, Recall=0.6000
n=40, seed=42, fold=2, Classical MLP: AUC=0.9250, Recall=0.8000
n=40, seed=42, fold=2, Classical Residual: AUC=0.9250, Recall=0.8000
n=40, seed=42, fold=2, Quantum Residual: AUC=0.9500, Recall=0.8000
n=40, seed=123, fold=0, Classical MLP: AUC=0.9556, Recall=0.6000
n=40, seed=123, fold=0, Classical Residual: AUC=0.9333, Recall=0.6000
n=40, seed=123, fold=0, Quantum Residual: AUC=0.9778, Recall=0.6000
n=40, seed=123, fold=1, Classical MLP: AUC=0.8750, Recall=0.6000
n=40, seed=123, fold=1, Classical Residual: AUC=0.9500, Recall=0.8000
n=40, seed=123, fold=1, Quantum Residual: AUC=0.9500, Recall=0

In [ ]:
#Phase 8.10.3 — Residual Network Performance Summary

p810_per_seed = (
    p810_df
    .groupby(["n_samples", "seed", "model"])
    .agg(
        auc=("roc_auc", "mean"),
        recall=("recall", "mean"),
        precision=("precision", "mean"),
        f1=("f1", "mean"),
        specificity=("specificity", "mean"),
        runtime=("training_seconds", "sum")
    )
    .reset_index()
)

p810_summary = (
    p810_per_seed
    .groupby(["n_samples", "model"])
    .agg(
        auc_mean=("auc", "mean"),
        auc_std=("auc", "std"),
        recall_mean=("recall", "mean"),
        recall_std=("recall", "std"),
        precision_mean=("precision", "mean"),
        f1_mean=("f1", "mean"),
        specificity_mean=("specificity", "mean"),
        runtime_mean=("runtime", "mean")
    )
    .reset_index()
)

display(p810_summary.round(4))

p810_summary.to_csv(
    P810_DIR / "phase810_summary.csv",
    index=False
)


,n_samples,model,auc_mean,auc_std,recall_mean,recall_std,precision_mean,f1_mean,specificity_mean,runtime_mean
0,40,Classical MLP,0.9534,0.0425,0.7333,0.0000,0.9722,0.8228,0.9861,0.0522
1,40,Classical Residual,0.9568,0.0381,0.7556,0.0385,0.9315,0.8182,0.9599,0.0752
2,40,Quantum Residual,0.9673,0.0306,0.7333,0.0000,0.9500,0.8129,0.9738,26.4925
3,80,Classical MLP,0.9624,0.0272,0.7667,0.1000,0.9620,0.8352,0.9734,0.0503
4,80,Classical Residual,0.9768,0.0207,0.7778,0.0694,0.9667,0.8503,0.9796,0.0749
5,80,Quantum Residual,0.9756,0.0129,0.7889,0.0509,0.9667,0.8599,0.9796,54.2855
6,160,Classical MLP,0.9862,0.0093,0.7722,0.0694,0.9942,0.8610,0.9966,0.0572
7,160,Classical Residual,0.9867,0.0093,0.8222,0.0674,0.9824,0.8882,0.9901,0.0832
8,160,Quantum Residual,0.9881,0.0064,0.8500,0.0167,0.9883,0.9125,0.9933,118.6063


In [ ]:
#Phase 8.10.4 — Does the Quantum Branch Add Value?

paired = p810_df.pivot(
    index=["n_samples", "seed", "fold"],
    columns="model",
    values=["roc_auc", "recall", "f1"]
)

paired.columns = [
    f"{metric}__{model}"
    for metric, model in paired.columns
]

paired = paired.reset_index()

comparisons = [
    ("Quantum Residual", "Classical MLP"),
    ("Quantum Residual", "Classical Residual")
]

difference_records = []

for n_samples, subset in paired.groupby("n_samples"):

    for model_a, model_b in comparisons:

        record = {
            "n_samples": n_samples,
            "comparison": f"{model_a} - {model_b}"
        }

        for metric in ["roc_auc", "recall", "f1"]:

            diff = (
                subset[f"{metric}__{model_a}"]
                - subset[f"{metric}__{model_b}"]
            )

            record[f"{metric}_diff_mean"] = diff.mean()
            record[f"{metric}_diff_std"] = diff.std()

        difference_records.append(record)

p810_differences = pd.DataFrame(
    difference_records
)

display(p810_differences.round(4))

p810_differences.to_csv(
    P810_DIR / "phase810_differences.csv",
    index=False
)


,n_samples,comparison,roc_auc_diff_mean,roc_auc_diff_std,recall_diff_mean,recall_diff_std,f1_diff_mean,f1_diff_std
0,40,Quantum Residual - Classical MLP,0.0139,0.0277,0.0000,0.0000,-0.0099,0.0511
1,40,Quantum Residual - Classical Residual,0.0105,0.0167,-0.0222,0.0667,-0.0053,0.0897
2,80,Quantum Residual - Classical MLP,0.0132,0.0249,0.0222,0.0667,0.0247,0.0447
3,80,Quantum Residual - Classical Residual,-0.0013,0.0155,0.0111,0.0601,0.0096,0.0514
4,160,Quantum Residual - Classical MLP,0.0019,0.0047,0.0778,0.1121,0.0515,0.0826
5,160,Quantum Residual - Classical Residual,0.0013,0.0047,0.0278,0.1149,0.0242,0.0822


In [ ]:
#8.10.5 — Quantum Contribution Consistency

consistency_records = []

for n_samples, subset in paired.groupby("n_samples"):

    for baseline in [
        "Classical MLP",
        "Classical Residual"
    ]:

        for metric in ["roc_auc", "recall", "f1"]:

            diff = (
                subset[f"{metric}__Quantum Residual"]
                - subset[f"{metric}__{baseline}"]
            )

            consistency_records.append({
                "n_samples": n_samples,
                "baseline": baseline,
                "metric": metric,
                "quantum_wins": int((diff > 0).sum()),
                "ties": int((diff == 0).sum()),
                "classical_wins": int((diff < 0).sum()),
                "mean_difference": diff.mean()
            })

p810_consistency = pd.DataFrame(
    consistency_records
)

display(p810_consistency.round(4))

p810_consistency.to_csv(
    P810_DIR / "phase810_consistency.csv",
    index=False
)


,n_samples,baseline,metric,quantum_wins,ties,classical_wins,mean_difference
0,40,Classical MLP,roc_auc,4,4,1,0.0139
1,40,Classical MLP,recall,0,9,0,0.0000
2,40,Classical MLP,f1,1,6,2,-0.0099
3,40,Classical Residual,roc_auc,3,6,0,0.0105
4,40,Classical Residual,recall,0,8,1,-0.0222
5,40,Classical Residual,f1,2,6,1,-0.0053
6,80,Classical MLP,roc_auc,5,2,2,0.0132
7,80,Classical MLP,recall,3,5,1,0.0222
8,80,Classical MLP,f1,4,4,1,0.0247
9,80,Classical Residual,roc_auc,2,2,5,-0.0013


In [ ]:
#Phase 8.10.6 — Export Results

import zipfile

zip_path = Path("phase810_results.zip")

with zipfile.ZipFile(
    zip_path, "w", zipfile.ZIP_DEFLATED
) as archive:

    for file in P810_DIR.glob("phase810_*.csv"):
        archive.write(
            file,
            arcname=file.name
        )

print("Saved:", zip_path)

from google.colab import files
files.download(str(zip_path))


Saved: phase810_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>